# NEXUS — train the dual-sensor mutation→cell model on human PPIs

NEXUS scores a mutation with **two orthogonal structural sensors** + a regulatory sign:
- **Intrinsic** — does it still *fold*? (folding ΔΔG)  ·  fixed, blind-validated
- **Extrinsic** — can it still *dock*? (interface ΔΔG + **dMaSIF geodesic surface**)  ·  **the trainable part**
- **Regulatory sign** — ON or OFF switch? (break an inhibitory brake → **GOF**)  ·  annotation

The trainable component is the **dMaSIF surface model**. It trains on interface **geometry** (self-supervised on
cross-chain contacts — *no labels*), so it can consume every protein-protein complex in the PDB.

### Data: how many PPIs, and where
| Layer | Count | Source |
|---|---|---|
| Binary PPIs (high-quality) | ~53,000 | HuRI |
| **Heteromeric complexes in the PDB** (≤3Å) ← the fetcher pulls these | **~34,000** | RCSB (auto-fetched below) |
| PPIs with *measured* ΔΔG (labels) | ~345 complexes | SKEMPI |
| Monomer structures (intrinsic sensor) | ~all 20,000 | AlphaFold DB |

**345 (SKEMPI) is only the *labelled* set.** The surface sensor needs no labels, so this notebook **auto-fetches**
thousands of real complexes straight from the PDB.


## 1 · GPU (Runtime → Change runtime type → GPU for the big run)


In [ ]:
import torch
device='cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '|', torch.cuda.get_device_name(0) if device=='cuda' else 'CPU (fine for the quick test)')


## 2 · Install the real tools (APBS + PDB2PQR + scikit-image)


In [ ]:
import sys, subprocess, shutil
subprocess.run('apt-get -qq install -y apbs > /dev/null 2>&1', shell=True)
subprocess.run(sys.executable+' -m pip -q install pdb2pqr scikit-image biopython cobra scikit-learn scipy 2>/dev/null', shell=True)
print('apbs:', shutil.which('apbs'), '| pdb2pqr:', shutil.which('pdb2pqr'))


## 3 · Get the code


In [ ]:
import os, sys
BRANCH='claude/vectorize-gex-propensity-zp09w8'
if not os.path.isdir('/content/cell'):
    os.system(f'git clone -q -b {BRANCH} https://github.com/nikku03/cell.git /content/cell')
else:
    os.system('cd /content/cell && git pull -q')   # pick up latest fixes on re-run
sys.path.insert(0,'/content/cell/colab')
print('code:', 'ok' if os.path.isdir('/content/cell/colab') else 'MISSING (private repo? add a token to the clone URL)')


## 4 · Quick working-check — auto-fetch 8 complexes, train, run the sensor stack (~1–2 min CPU)
`fetch:N` pulls N real protein-protein complexes from the PDB, then trains dMaSIF (held out by complex) and
exercises the full sensor stack.


In [ ]:
import nexus_train, importlib; importlib.reload(nexus_train)
res = nexus_train.fetch_and_train(8, cache='/content/pdb_cache', epochs=15, device=device)
print('\nWORKING-CHECK:', 'PASS' if res.get('working_check') else 'FAIL', '| held-out dMaSIF AUC', res.get('dmasif_heldout_auc'))


## 5 · Google Drive — persist everything, with timely auto-save (survives a disconnect)
Mount Drive **before** the big run. This cell:
- **resumes** any clouds already saved in Drive (the build then skips them — no re-computing APBS),
- starts a background thread that copies every new cloud + the model + the results JSON to Drive **every 90 s**.

So a Colab disconnect loses at most ~90 s of work. Cloud writes are atomic, so the sync never copies a half-written file.
Re-running the whole notebook after a drop picks up exactly where it stopped.

In [ ]:
# ── Google Drive: durable cache + timely auto-save ──
from google.colab import drive
import os, shutil, threading, time, glob
drive.mount('/content/drive')

LOCAL = '/content/pdb_cache'                     # fast local cache the build writes to
DRIVE = '/content/drive/MyDrive/nexus_cache'     # durable Drive copy
os.makedirs(f'{LOCAL}/clouds', exist_ok=True)
os.makedirs(f'{DRIVE}/clouds', exist_ok=True)

# RESUME: pull clouds/model already in Drive back to local, so the build skips them
restored = 0
for f in glob.glob(f'{DRIVE}/clouds/*.pkl'):
    dst = f'{LOCAL}/clouds/{os.path.basename(f)}'
    if not os.path.exists(dst):
        shutil.copy2(f, dst); restored += 1
if os.path.exists(f'{DRIVE}/nexus_dmasif.pt') and not os.path.exists('/content/nexus_dmasif.pt'):
    shutil.copy2(f'{DRIVE}/nexus_dmasif.pt', '/content/nexus_dmasif.pt')
print(f'resumed {restored} clouds from Drive (build will skip these)')

# AUTO-SAVE: background thread mirrors new clouds + model + results to Drive every 90 s
def _sync():
    while True:
        try:
            n = 0
            for f in glob.glob(f'{LOCAL}/clouds/*.pkl'):
                dst = f'{DRIVE}/clouds/{os.path.basename(f)}'
                if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(f):
                    shutil.copy2(f, dst); n += 1
            for src, dst in [('/content/nexus_dmasif.pt', f'{DRIVE}/nexus_dmasif.pt'),
                             ('/content/cell/outputs/orphan/nexus_train.json', f'{DRIVE}/nexus_train.json')]:
                if os.path.exists(src):
                    shutil.copy2(src, dst)
            if n:
                print(f'  [autosave] +{n} new clouds -> Drive ({time.strftime("%H:%M:%S")})', flush=True)
        except Exception as e:
            print('  [autosave] skipped:', e, flush=True)
        time.sleep(90)
threading.Thread(target=_sync, daemon=True).start()
print(f'auto-save on: {DRIVE}  (clouds + model + results every 90 s)')

## 6 · Scale up — 5,000 human complexes (parallel + resumable, saving to Drive)
The surface/APBS build runs **across all CPU cores** and **caches each complex to disk**, and the auto-save cell above
mirrors every new cloud to Drive. So:
- a disconnect never loses built clouds — Drive keeps them, and re-running resumes where it left off,
- you pay the APBS cost **once per complex, ever**.

On a 12-core box: ~**0.8 s/complex effective**, so ~**1 hr for 5,000** (~**7–9 hr for all ~34,000**).
`human_only=True` restricts to Homo sapiens. The GPU is idle during the build (APBS is CPU) and only trains at the end.

In [ ]:
# 5,000 human complexes (near the accuracy ceiling; ~1 hr on 12 cores; clouds auto-saved to Drive):
res = nexus_train.fetch_and_train(5000, cache='/content/pdb_cache', epochs=60, device=device, human_only=True)
print('trained on', res.get('n_usable'), 'complexes | held-out dMaSIF AUC', res.get('dmasif_heldout_auc'))

# the full firehose (~34k, ~7-9 hr, resumable — safe to re-run if the session drops):
# res = nexus_train.fetch_and_train(40000, cache='/content/pdb_cache', epochs=60, device=device)
# every built cloud lives in Drive (nexus_cache/clouds), so a re-run after a disconnect skips them all.

## 7 · Also train on the *labelled* SKEMPI set, or your own list
SKEMPI gives measured ΔΔG for the binding-node validation; the fetcher gives unlimited unlabelled surfaces.

In [ ]:
# labelled SKEMPI complexes (measured ΔΔG):
import urllib.request, csv
urllib.request.urlretrieve('https://life.bsc.es/pid/skempi2/database/download/skempi_v2.csv', '/content/skempi.csv')
ALL=sorted({r[0].split('_')[0] for r in list(csv.reader(open('/content/skempi.csv'),delimiter=';'))[1:] if r and r[0]})
print(len(ALL),'SKEMPI complexes')
# res = nexus_train.main(ALL, cache='/content/pdb_cache', epochs=60, device=device)
# or your own complex PDB ids (incl. AlphaFold-Multimer-predicted, saved into the cache):
# res = nexus_train.main(['4HFK','1DVF', ...], cache='/content/pdb_cache', epochs=40, device=device)


## What you get — and the honest edges
- **Trained:** the dMaSIF geodesic **surface (extrinsic) sensor**, saved to `outputs/nexus_dmasif.pt` — scales with data.
- **Fixed:** intrinsic **stability** node (blind-validated) + **regulatory-sign** GOF layer (annotation).
- **Limits (from the validation):** the binding sensor needs *complex* structures (the ~34k here, or AlphaFold-Multimer
  for the rest); the FBA→phenotype step is a *demonstration*, not a validated predictor; *neomorphic* GOF is out of
  reach; the regulatory sign is high-precision but annotation-recall-limited.
